# 02 — Case study (Section 4)

Reliability and resilience of the two industrial structures of the paper: a steel braced frame (BF,
`fixed`) and a base-isolated building (BI, `isolated`), at an Italian site.

| Section | Paper |
|---|---|
| 1. Inputs: hazard, state-dependent fragilities, recovery times | Sec. 4.1–4.2, Fig. 9 |
| 2. Generator matrices $Q$ | Sec. 4.2 |
| 3. Long-term state probabilities | Sec. 4.3.1, Fig. 10, Table 4 |
| 4. Reliability: $F_\mathcal{T}$ and $\beta$ vs $\mu_{10}$ | Sec. 4.3.2, Fig. 11 |
| 5. Resilience: $\mathcal{R}_0$, $\rho$, spectral approximation; $(\beta,\rho)$ plane | Sec. 4.3.3, Table 5, Figs. 12–13 |

Figures are written to `output/case_study/` (compare them with `figures/case_study/` using
`python tests/compare_figures.py`).

> **Data availability.** The case-study data are confidential and not included in the public
> repository (available upon request, see `README.md`). Without them this notebook stops at the
> first cell that needs them, with a message explaining what is missing.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import expm

from ctmc_pbee import core, metrics        # CTMC model + parametric studies, metrics (Eq. 10-19)
from ctmc_pbee import generator           # Q matrices from hazard + fragilities + recovery
from ctmc_pbee import plotting
from ctmc_pbee import utils
from ctmc_pbee.paths import DATA, output_dir, require_data

In [ ]:
general = {'savefig': False,
           'mutefig': True,
           'savepkl': True,
           'createFolder': True,
           'appDate': '20260512',
           'JupyterFilename': '14-EESD-j1-ctmc-revised-state-dep.ipynb'}

In [ ]:
# Plotting setup for seismic hazard, damage states, and fragility
# Example templates for line styles and colors
# style = hazard_styles['medium']
# plt.plot(x, y, linestyle=style['linestyle'], color=style['color'], label=style['label'])

# Hazard settings
hazard_styles = {
    'low':    {'linestyle': '--', 'label': 'Low Seismicity',    'color': '#FC9F5C'},
    'medium': {'linestyle': '-.', 'label': 'Medium Seismicity', 'color': '#ECE4B6'},
    'high':   {'linestyle': ':',  'label': 'High Seismicity',   'color': '#33C97F'},
}

# Damage state line styles and labels
damage_styles = {
    'DS0': {'linestyle': '-',  'label': '$DS_0$', 'color': '#08407F'},
    'DS1': {'linestyle': '-.', 'label': '$DS_1$', 'color': '#08407F'},
    'DS2': {'linestyle': '--', 'label': '$DS_2$', 'color': '#08407F'},
}

# Fragility colors and linestyles
fragility_styles = {
    'color': {
        'BF':  '#4A6378',
        'ISO': '#660033',
        'glo': '#414141',
    },
    'linestyle': {
        'mean': '-',
        'perc': '-.',
    }
}

# Structural type color schemes
struct_colors = {
    'swi': {
        'low':    {'color': '#77355B', 'label': 'low',   'linestyle': '-'},
        'medium': {'color': '#B35289', 'label': 'medium','linestyle': '-'},
        'high':   {'color': '#D094B6', 'label': 'high',  'linestyle': '-'},
    },
    'ita': {
        'low':    {'color': '#08407F', 'label': 'low',   'linestyle': '-'},
        'medium': {'color': '#0C62C0', 'label': 'medium','linestyle': '-'},
        'high':   {'color': '#816EF6', 'label': 'high',  'linestyle': '-'},
    }
}

struct_colors2 = {
    'isolated': { 'color': '#77355B', 'label': 'isolated',   'linestyle': '-' },
    'fixed'   : { 'color': '#08407F', 'label': 'fixed',   'linestyle': '-' }
}



## 1. Inputs: hazard, state-dependent fragilities, recovery times (Sec. 4.1–4.2)

## a) loading 

Note: here you need to assign the pkl file where the fragilities and hazard are stored.

In [ ]:
pklFilename = f"SS-input-hazard-fragilities-20260512.pkl"
# confidential case-study data, available upon request (README, 'Data availability')
require_data(pklFilename, 'recovery.json')

general['inputLoading'   ] = True
general['pkl2load'       ] = True
general['JupyterFilename'] = '14-SS-j1-ctmc.ipynb'
general['dataFolder'     ] = DATA                       # <repo>/data
general['figFolder'      ] = output_dir('case_study')   # <repo>/output/case_study

print(f"# Hazard, state-dependent fragilities and recovery rates are required.")
print(f"# If they were already computed and stored, set 'inputLoading' to True and 'pkl2load' to True, and assign the pkl file where they are stored.")
print(general['dataFolder'] / pklFilename)

if general['inputLoading'] is True: 
    # it means that all the hazard, state-dependent fragilities and recovery were already computed and stored, so now it's just a matter of loading them
    if general['pkl2load'    ] is True:
        with open(general['dataFolder'] / pklFilename, 'rb') as f:
            data_input = pickle.load(f)
            hazard     = data_input['hazard']
            fixed      = data_input['pb_transitions_exc_fixed']
            isolated   = data_input['pb_transitions_exc_isolated']
            with open(general['dataFolder'] / 'recovery.json', "r") as f:
                recovery_dict = json.load(f)
            print('--> Loaded hazard curves, fragilities and recoveries')
            print('--> Dictionary structure:')
            print(f"----> Hazard: \n        {utils.get_keys_by_level(hazard)}")
            print(f"----> State-dependent fragilities - fixed: \n        {utils.get_keys_by_level(fixed)}")
            print(f"-----> Recovery rates - fixed: \n        {utils.get_keys_by_level(recovery_dict['fixed'])}")
            print(f"----> State-dependent fragilities - isolated: \n        {utils.get_keys_by_level(isolated)}")
            print(f"-----> Recovery rates - isolated: \n        {utils.get_keys_by_level(recovery_dict['isolated'])}")




Figure 9 — median state-dependent fragilities, BF vs BI:

In [ ]:

fig, axes = plotting.plot_median_fragilities(
    pga            = fixed['pb_transitions_exc']['DS0|DS0']['pga'],
    ds_probs_bf    = fixed['pb_transitions_exc'],
    ds_probs_iso   = isolated['pb_transitions_exc'],
    struct_colors2 = struct_colors2,
    general        = general        # <-- use the full general dict, not {'muteFig': False}
)

fname = '03-BF-vs-ISO-median_comparison.svg'
plotting.save_svg(general, fname)
plt.show()

In [ ]:

# structConf = 'fixed' # 'fixed' , 'isolated'
# print(f"--> State-dependent fragilities for the {structConf} case:")
# figStateDep, axStateDep = plotting.plot_fragilities(fixed['pb_transitions_exc'], col_frag='blue', fragil_ls={'mean':'-', 'perc':'--'},
#                              general={'muteFig': False})
# plt.show()
# structConf = 'isolated' # 'fixed' , 'isolated'
# print(f"--> State-dependent fragilities for the {structConf} case:")
# figStateDep, axStateDep = plotting.plot_fragilities(isolated['pb_transitions_exc'], col_frag='red', fragil_ls={'mean':'-', 'perc':'--'},
#                              general={'muteFig': False})
# plt.show()

structConf = 'fixed' # 'fixed' , 'isolated'
print(f"--> State-dependent fragilities for the {structConf} case version 2:")
figStateDep, axStateDep = plotting.plot_fragilities_v2(fixed['pb_transitions_exc'], col_frag='blue', fragil_ls={'mean':'-', 'perc':'--'},
                             general={'muteFig': True}, struct_color=struct_colors2['fixed'])
plt.show()

structConf = 'isolated' # 'fixed' , 'isolated'
print(f"--> State-dependent fragilities for the {structConf} case version 2:")
figStateDep, axStateDep = plotting.plot_fragilities_v2(isolated['pb_transitions_exc'], col_frag='red', fragil_ls={'mean':'-', 'perc':'--'},
                             general={'muteFig': True}, struct_color=struct_colors2['isolated'])
plt.show()

print(f"--> Seismic Hazard:")
figHazard, axHazard = plotting.plot_hazard(hazard, list(hazard_styles.values()))
plt.show()

print(f"--> Recovery Table [days]:")
recovery_table = pd.DataFrame(recovery_dict)
print(recovery_table)

In [ ]:
ds_probs_dict = {
    'isolated': isolated['pb_transitions_exc'],
    'fixed'   : fixed['pb_transitions_exc']
}

fig, axes = plotting.plot_fragilities_bands(
    ds_probs_dict = ds_probs_dict,
    general       = {'muteFig': True},
    struct_colors = struct_colors2
)

## 2. Generator matrices $Q$

## Compute upper triangular Q matrix
Consider different quantiles in the hazard and in the state-dependent fragilities

In [ ]:

Qseism, _, __, df_Qseism= generator.seismic_rates(
                            lambda_im=hazard['ita'], 
                            im_axis=hazard['ita']['PGA'], 
                            pb_transitions=fixed['pb_transitions_exc'])

# print(df_Qseism)

Note that here $\lambda_{00}$ and $\lambda_{11}$ are not the real ones. Indeed, here they are computed as $\int P(DS\mid DS, IM)\,|d\lambda|$ instead of $\lambda_{00}=-(\lambda_{01}+\lambda_{02})$ and $\lambda_{11}=-(\mu_{10}+\lambda_{12})$.


In [ ]:
utils.get_keys_by_level(Qseism)
# Qseism['ita']['lambda_0.5']['fixed']['p50']
Qseism['lambda_0.5']['p50']

## Compute full Q matrix

- **Qfull**
  - hazard location
    - hazard quantile
      - structure type
        - PB quantile (p01, …, p99)
          - recovery level (low, medium, high)
            - Q matrix


In [ ]:
# Qsim['ita']['lambda_0.5']['fixed']['p50']['R2']
# Qsim['ita']['lambda_0.5']['isolated']['p50']['R1']
recovery_dict

In [ ]:
hazard_locations = ['ita'] #, 'swi']
hazard_qtiles = ['lambda_0.05', 'lambda_0.5', 'lambda_0.95']
struct_dicts = {'fixed': fixed, 'isolated': isolated}
pb_transition_qtiles = ['p01','p05','p16','p50','p84','p95','p99']
recovery_dict = recovery_dict

Qsim = generator.build_Q_matrices(     
    hazard= hazard,
    hazard_locations= ['ita'],
    hazard_qtiles = ['lambda_0.05', 'lambda_0.5', 'lambda_0.95'],
    struct_dicts = {'fixed': fixed, 'isolated': isolated},
    pb_transition_qtiles= ['p01','p05','p16','p50','p84','p95','p99'],
    recovery_dict = recovery_dict,
    verbose = True
)

# print(Qsim)
utils.get_keys_by_level(Qsim)

In [ ]:
print(Qsim['ita']['lambda_0.5']['isolated']['p50']['R1'][0,1])
print(Qsim['ita']['lambda_0.5']['isolated']['p95']['R1'][0,1])
print(Qsim['ita']['lambda_0.5']['isolated']['p05']['R1'][0,1])

# print(fixed['pb_transitions_exc'])

## 3. Long-term state probabilities — Figure 10, Table 4 (Sec. 4.3.1)

In [ ]:
general['figFolder']

In [ ]:

# =========================
# General settings
# =========================
general.update({
    'savefig': True,
    'mutefig': True
})


debugQ = False
Thorizon = 50     # years
max_time = 475    # years
state_labels = ['DS0', 'DS1', 'DS2']
rate_levels = ['low', 'high']

# Dense time grid
t_vals = np.linspace(0, max_time, max_time + 1)
P0 = np.array([1.0, 0.0, 0.0])

# =========================
# Build all possible Q-matrices
# =========================
Qsim = generator.build_Q_matrices(
    hazard              = hazard,
    hazard_locations    = ['ita'],
    hazard_qtiles       = ['lambda_0.05', 'lambda_0.5', 'lambda_0.95'],
    struct_dicts        = {'fixed': fixed, 'isolated': isolated},
    pb_transition_qtiles= ['p01','p05','p16','p50','p84','p95','p99'],
    recovery_dict       = {
                            'fixed':    {'low': 30,     'high': 365},
                            'isolated': {'low': 1,   'high': 50}
                          },
    verbose = False
)

# =========================
# Preparation of result containers
# =========================
results_table = {}
results_Pt    = {}
Q_full        = utils.recursive_defaultdict()


# =========================
# Main loop for fixed hazard and quantiles of the seismic rate
# =========================
hqtile  = 'lambda_0.05' # lambda_0.05
stqtile = 'p84'         # p84

for hl in hazard_locations:

    results_table[hl] = {s: {lvl: None for lvl in rate_levels} for s in struct_dicts}
    results_Pt[hl]    = {s: {lvl: None for lvl in rate_levels} for s in struct_dicts}

    for st in struct_dicts.keys():

        # Extract Q matrices for recovery levels
        Q_low  = Qsim[hl][hqtile][st][stqtile]['low']
        Q_high = Qsim[hl][hqtile][st][stqtile]['high']
        Q_matrices = {'low': Q_low, 'high': Q_high}

        if debugQ is True:
            print(Q_low, Q_high)
            print("mu_low =",  Q_low[1, 0], "mu_high =", Q_high[1, 0])
            print("seismic rates:", Q_low[0,1], Q_low[0,2], Q_low[1,2])

        # Plot
        fig, axes = plt.subplots(2, figsize=(3.5, 3), sharex=True)

        for rec_level, ax in zip(rate_levels, axes):
            Qtemp = Q_matrices[rec_level]
            
            # recovery rate in days
            recovery_rate_in_days = (np.log(2) / Qtemp[1, 0]) * 365
            print(f"Structural case: {st} – Recovery in = {recovery_rate_in_days:.1f} days")

            # Debug print
            if debugQ is True:
                print(f"\n--- DEBUG Q ({rec_level}) ---")
                print("t_vals (years):", t_vals[:5])
                print(Qtemp)
                print("mu =", Qtemp[1,0])
                print("seismic rates:", Qtemp[0,1], Qtemp[0,2], Qtemp[1,2])

            # Compute CTMC evolution
            P_t = np.array([P0 @ expm(Qtemp * t) for t in t_vals])

            # Store P(t)
            results_Pt[hl][st][rec_level] = P_t
            results_table[hl][st][rec_level] = P_t[Thorizon].round(2).tolist()

            # Intersection p50 line
            t_cross, p_cross = plotting.plt_intersection_p50(P_t, t_vals)  # ← aggiunta
            if t_cross is not None:
                print(f"  → Intersection at t = {t_cross:.1f} years, p = {p_cross:.2f}")
            else:
                print(f"  → No intersection found within [{t_vals[0]:.0f}, {t_vals[-1]:.0f}] years")


            # Save full Q
            Q_full[hl][hqtile][st][stqtile][rec_level] = Qtemp

            # Get color/linestyle configuration
            hazard_col = struct_colors['swi' if hl == 'swi' else 'ita'][rec_level]

            # Plot DS probabilities
            for idx, lab in enumerate(state_labels):
                ax.plot(
                    t_vals, P_t[:, idx],
                    linestyle = hazard_col['linestyle'],
                    color     = hazard_col['color'],
                    label     = hazard_col['label']
                )

            # Mark intersection
            if t_cross is not None:
                ax.axvline(t_cross, color="darkgrey", linestyle="--", alpha=0.7)
                ax.plot(t_cross, p_cross, 'x', color="black", markersize=8, mew=2)

            ax.grid(True)
            ax.set_xlim([0, max_time])

        axes[-1].set_xlabel("Time [t]")
        plt.tight_layout()


        # Hide y-labels
        for ax in axes:
            plt.setp(ax.get_yticklabels(), visible=False)

        if general['savefig']:
            fname = f"03m-state-probabilities-{hl}-lambda_095-{st}-mean.svg"
            plotting.save_svg(general, fname.replace('..','.'))

        plt.show()


# ── Popola Q_full per tutti i quantili da Qsim ──────────────────────────────
for hl in hazard_locations:
    for hq in ['lambda_0.05', 'lambda_0.5', 'lambda_0.95']:
        for st in struct_dicts.keys():
            for sq in ['p01','p05','p16','p50','p84','p95','p99']:
                for rec in ['low', 'high']:
                    Q_full[hl][hq][st][sq][rec] = Qsim[hl][hq][st][sq][rec]

# reset ai valori di default per le celle successive
hqtile  = 'lambda_0.05'
stqtile = 'p84'


# =========================
# Results table → DataFrame
# =========================
rows = []
for hl in results_table:
    fixed_vals = [results_table[hl]['fixed'][lvl]    for lvl in rate_levels]
    isol_vals  = [results_table[hl]['isolated'][lvl] for lvl in rate_levels]
    rows.append(fixed_vals + isol_vals)

df_results = pd.DataFrame(
    rows,
    index=results_table.keys(),
    columns=pd.MultiIndex.from_product(
        [['fixed', 'isolated'], ['mu_low', 'mu_high']]
    )
)

print(df_results)


## 4. Reliability — Figure 11 (Sec. 4.3.2)

Failure probability $F_\mathcal{T}$ and reliability index $\beta$ as functions of the recovery rate $\mu_{10}$, with the seismic rates fixed.

In [ ]:
utils.get_keys_by_level(Q_full)
stqtile = 'p50' # 'p84'
hqtile = 'lambda_0.05' # 'lambda_0.05', 'lambda_0.5', 'lambda_0.95'
Q_full['ita'][hqtile]['fixed'][stqtile]['low']
hqtile, stqtile

In [ ]:
T_hor = 1 # years
Qtest1 = Q_full['ita'][hqtile]['fixed'][stqtile]['low']
mus = [8.43, 2.10, 0.69] # [1/years] for fixed case

for mu in mus:
    F_tau, surv = metrics.F_tau(core.extract_Q_T(core.set_recovery_rate(Qtest1, mu)), T_hor)
    beta = metrics.beta_index(F_tau)
    print(f"For mu  = {mu:.2f} 1/years braced frame in T_hor {T_hor}:")
    print(f"F_tau(T={T_hor}) = {F_tau:.4f}, beta = {beta:.4f}")
#print(f"F_tau(T={T_hor}) = {F_tau:.4f}, survival = {surv:.4f}, rates = {rates}")
#%print(f"Beta index = {beta:.4f}")

print("\n ################ ")
Qtest2 = Q_full['ita'][hqtile]['isolated'][stqtile]['high']
mus = [253,8.43,5.05] # [1/years] for fixed case

for mu in mus:
    F_tau, surv = metrics.F_tau(core.extract_Q_T(core.set_recovery_rate(Qtest2, mu)), T_hor)
    beta = metrics.beta_index(F_tau)
    print(f"For mu  = {mu:.2f} 1/years braced frame in T_hor {T_hor}:")
    print(f"F_tau(T={T_hor}) = {F_tau:.4f}, beta = {beta:.4f}")
# print(f"F_tau(T={T_hor}) = {F_tau:.4f}, survival = {surv:.4f}, rates = {rates}")
# print(f"Beta index = {beta:.4f}")


In [ ]:
general['savefig'] = True
general['mutefig'] = False
hqtile, stqtile

In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as colors
from matplotlib.colors import LinearSegmentedColormap, to_rgba

mus   = np.logspace(-1, 3, 50)
thors = np.array([1.0, 10, 50.0])

Qtest1 = Q_full['ita'][hqtile]['fixed'][stqtile]['high']
Qtest2 = Q_full['ita'][hqtile]['isolated'][stqtile]['high']


normCol = colors.LogNorm(vmin=thors.min(), vmax=thors.max())
cmap = cm.viridis

normCol = colors.LogNorm(vmin=thors.min(), vmax=thors.max())
darkisolated = struct_colors2['isolated']['color'] # '#77355B'
r, g, b, _ = to_rgba(darkisolated)
# Create colormap: same color, alpha from 0 → 1
darkisolated_alpha = LinearSegmentedColormap.from_list(
    "darkisolated",
    [(r, g, b, 1.0),
     (r, g, b, 0.4)],
    N=256
)
cmap_iso = darkisolated_alpha #plt.cm.viridis
darkfixed   = struct_colors2['fixed']['color'] # '#08407F'
r, g, b, _ = to_rgba(darkfixed)
# Create colormap: same color, alpha from 0 → 1
darkfixed_alpha = LinearSegmentedColormap.from_list(
    "darkfixed",
    [(r, g, b, 1.0),
     (r, g, b, 0.4)],
    N=256
)
cmap_fix = darkfixed_alpha #plt.cm.viridis
MarkerSize = 4


fig, axes = plt.subplots(1, 2, figsize=(9, 2.5))
axF, axB = axes  # unpack for clarity
# plt.figure(figsize=(10, 5))

for T_hor in thors:
    betas_iso, betas_fix = [], []
    Fs_iso, Fs_fix = [], []

    for mu in mus:
        F_iso, _ = metrics.F_tau(core.extract_Q_T(core.set_recovery_rate(Qtest2, mu)), T_hor)
        beta_iso = metrics.beta_index(F_iso)
        betas_iso.append(beta_iso), Fs_iso.append(F_iso)

        F_fix, _ = metrics.F_tau(core.extract_Q_T(core.set_recovery_rate(Qtest1, mu)), T_hor)
        beta_fix = metrics.beta_index(F_fix)
        betas_fix.append(beta_fix), Fs_fix.append(F_fix)

    color_iso = cmap_iso(normCol(T_hor))
    color_fix = cmap_fix(normCol(T_hor))

    axB.semilogx(mus, betas_iso, color=color_iso, linestyle='-')
    axB.semilogx(mus, betas_fix, color=color_fix, linestyle='-')

    axF.semilogx(mus, Fs_iso, color=color_iso, linestyle='-')
    axF.semilogx(mus, Fs_fix, color=color_fix, linestyle='-')

    # add the points for the reference mus
    ref_mus_iso = [253,8.43,5.05]
    ref_mus_fix = [8.43, 2.10, 0.69]
    for mu in ref_mus_iso:
        F_iso, _ = metrics.F_tau(core.extract_Q_T(core.set_recovery_rate(Qtest2, mu)), T_hor)
        beta_iso = metrics.beta_index(F_iso)
        axB.plot(mu, beta_iso, 'o', color=color_iso, markersize=MarkerSize)
        axF.plot(mu, F_iso, 'o', color=color_iso, markersize=MarkerSize)
    for mu in ref_mus_fix:
        F_fix, _ = metrics.F_tau(core.extract_Q_T(core.set_recovery_rate(Qtest1, mu)), T_hor)
        beta_fix = metrics.beta_index(F_fix)
        axB.plot(mu, beta_fix, 'o', color=color_fix, markersize=MarkerSize)
        axF.plot(mu, F_fix, 'o', color=color_fix, markersize=MarkerSize)



axF.grid(True, which="both", ls="--", alpha=0.5)
axB.grid(True, which="both", ls="--", alpha=0.5)
axB.set_ylim([0, 3.5])

# # # mute to comment / uncomment
# axF.set_xticklabels([])
# axF.set_yticklabels([])
# axB.set_xticklabels([])
# axB.set_yticklabels([])


plt.tight_layout()
plt.box(True)
plt.setp(ax.spines.values(), linewidth=1.5)


# plt.xlabel(r"$\mu$")
# plt.ylabel(r"$\beta$")
plt.tight_layout()

if general['savefig']:
    fname = f"IC-ftau-beta-function-mu10.svg"
    plt.savefig(general['figFolder'] / fname, transparent=True, format='svg', bbox_inches='tight')
    print(f"Figure saved as {fname} in {general['figFolder']}")

plt.show()


In [ ]:
general['savefig'] = True


## 5. Resilience — Table 5, Figures 12–13 (Sec. 4.3.3)

Conditional resilience metric $\mathcal{R}_0$ (Eq. 18), its spectral approximation $\mathcal{R}_0 \approx 1 - w_0 q_0$ (Eq. 27) and the resilience index $\rho$ (Eq. 19), computed with `core.reliability_resilience_summary`.

In [ ]:
# Settings used by the resilience cells below
general.update({'saveFolder': general['figFolder'], 'saveFig': True, 'muteFig': True})

In [ ]:
# Settings of Table 5
hqtile  = 'lambda_0.05'   # hazard quantile
stqtile = 'p50'           # fragility quantile
T_HORS  = [1.0, 50.0]     # time horizons [year]

# 'low' recovery level  = short repair = high mu10
# 'high' recovery level = long  repair = low  mu10
SCENARIO_MAP = {'short': 'low', 'long': 'high'}

# Qsim was built in Section 3 (same hazard, fragilities and recovery times of Table 3)

# Sanity check: print rates for hqtile / stqtile
print(f'\nQ matrices — hqtile={hqtile}, stqtile={stqtile}')
for st in ['fixed','isolated']:
    for lvl in ['low','high']:
        Q = Qsim['ita'][hqtile][st][stqtile][lvl]
        mu10 = Q[1,0]; tmed = np.log(2)/mu10*365
        print(f'  {st:8s} {lvl:5s}: mu10={mu10:.4f} (tmed={tmed:.1f}d)  '
              f'lam01={Q[0,1]:.4e}  lam02={Q[0,2]:.4e}  lam12={Q[1,2]:.4e}')


### Table 5

In [ ]:
results = {'fixed': {}, 'isolated': {}}

for paper_st in ['fixed', 'isolated']:
    for sc_label, nb_lvl in SCENARIO_MAP.items():
        Q = Qsim['ita'][hqtile][paper_st][stqtile][nb_lvl]
        for T_hor in T_HORS:
            r = core.reliability_resilience_summary(Q, T_hor)
            results[paper_st][(sc_label, T_hor)] = r
            print(f'{paper_st:8s} {sc_label:5s} T={T_hor:4.0f}yr | '
                  f'mu10={r["mu10"]:.2f} | beta={r["beta"]:.3f} | '
                  f'R0={r["R0"]:.4f} | rho={r["rho"]:.3f} | '
                  f'R0_sp={r["R0_sp"]:.4f} | dR0={r["dR0"]:.2e}')


Formatted as in Table 5 (the paper reports T = 50 yr):

In [ ]:
utils.print_metrics_table(results['fixed'],    'BRACED FRAME  (Table 5)', SCENARIO_MAP, T_HORS)
utils.print_metrics_table(results['isolated'], 'BASE-ISOLATED (Table 5)', SCENARIO_MAP, T_HORS)


## 6. Figure 12 — $\mathcal{Q}_0$ and $\rho$ vs $\mu_{10}$

In [ ]:
# Figure 12 (fig_Q0_rho_aggregate) uses the plotting functions in
# src/ctmc_pbee/plotting.py (plot_Q0_rho).
#
# NOTE: in the original notebook this cell was an inline copy of that module whose
# `if __name__ == '__main__'` block rebuilt Qsim (same inputs as above) and changed the
# fragility quantile from 'p50' to 'p84'. Figure 12 was produced with 'p84', while
# Table 5 above uses 'p50'.  TODO: confirm against the paper which quantile is intended.
stqtile = 'p84'

In [ ]:
from ctmc_pbee.plotting import plot_Q0_rho


general['savefig'], general['mutefig'] = True, True

Qtest1 = Qsim['ita'][hqtile]['fixed'][stqtile]['high']     # BF  long recovery
Qtest2 = Qsim['ita'][hqtile]['isolated'][stqtile]['high']  # BI  long recovery
fig, axes = plot_Q0_rho(Qtest1, Qtest2, savefig=False, mutefig=general['mutefig'])  # saved in the next cell

In [ ]:
general['savefig'], general['mutefig'] = True, True
print(general['savefig'], general['mutefig'])
if general['savefig']:
    for general['mutefig'] in [False, True]:
        if general['mutefig']:
            axes[0].set_xlabel(''); axes[0].set_ylabel(''); axes[0].set_title('')
            axes[1].set_xlabel(''); axes[1].set_ylabel(''); axes[1].set_title('')
            axes[0].set_xticklabels([]); axes[0].set_yticklabels([])
            axes[1].set_xticklabels([]); axes[1].set_yticklabels([])
            
            leg1 = axes[0].get_legend()
            leg2 = axes[1].get_legend()
            if leg1 is not None: leg1.remove()
            if leg2 is not None: leg2.remove()
            savefig = 'mfig_Q0_rho_aggregate.svg'
        else:
            savefig = 'fig_Q0_rho_aggregate.svg'
        fig.savefig(general['saveFolder'] / savefig, bbox_inches='tight', transparent=True, format=savefig.split('.')[-1])
        print(f'Saved: {savefig}')

In [ ]:
colors_bf = ['#08407F', '#3b6db3', '#8ab0e0']
colors_bi = ['#77355B', '#b05289', '#d9a0c0']
ref_mus   = {'fixed': [8.43, 0.69], 'isolated': [252.99, 5.06]}


### Figure 13 — $(\beta, \rho)$ plane

In [ ]:
# ── Beta vs Rho parametric plot — stile plot_rho_beta_absolute_rates ──────────
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

mpl.rcParams.update({
    "font.size"        : 13,
    "axes.titlesize"   : 12,
    "axes.labelsize"   : 13,
    "xtick.labelsize"  : 11,
    "ytick.labelsize"  : 11,
    "mathtext.fontset" : "stix",
})

T_hor_plot = 50.0
N_mu       = 80
n_quad     = 300

mus_sweep = np.logspace(-1, 3, N_mu)

hqtile  = 'lambda_0.05'
stqtile = 'p50'

Q_bf_base = Qsim['ita'][hqtile]['fixed'   ][stqtile]['high'].copy()
Q_bi_base = Qsim['ita'][hqtile]['isolated'][stqtile]['high'].copy()

# ── Sweep curves ─────────────────────────────────────────────────────────────
betas_bf, rhos_bf = core.beta_rho_vs_mu10(Q_bf_base, mus_sweep, T_hor_plot, n_quad)
betas_bi, rhos_bi = core.beta_rho_vs_mu10(Q_bi_base, mus_sweep, T_hor_plot, n_quad)

# ── Reference points from results dict ───────────────────────────────────────
# results[paper_st][(sc_label, T_hor)] → r with keys beta, rho, mu10
ref_pts = {
    'fixed'   : {},
    'isolated': {},
}
for sc_label in SCENARIO_MAP:
    for st in ['fixed', 'isolated']:
        r = results[st][(sc_label, T_hor_plot)]
        ref_pts[st][sc_label] = (r['beta'], r['rho'], r['mu10'])

# ── Colors ───────────────────────────────────────────────────────────────────
col_bf  = '#08407F'
col_bi  = '#77355B'
cmap_bf = plt.cm.Blues
cmap_bi = plt.cm.RdPu

norm_mu = Normalize(vmin=np.log10(mus_sweep.min()),
                    vmax=np.log10(mus_sweep.max()))

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6.5))
fig.subplots_adjust(right=0.74)

x_min, x_max  = 0, 4
y_min, y_max  = 0, 4
beta_thresh   = 1.0   # reliability threshold — adjust to your target
rho_thresh    = 2.0   # resilience threshold  — adjust to your target

# # ── Quadrant shading ─────────────────────────────────────────────────────────
# ax.fill_between([beta_thresh, x_max], rho_thresh, y_max,
#                 alpha=0.07, color='steelblue',   zorder=1)   # ✓ resi  ✓ reli
# ax.fill_between([x_min, beta_thresh], rho_thresh, y_max,
#                 alpha=0.07, color='forestgreen', zorder=1)   # ✓ resi  ✗ reli
# ax.fill_between([x_min, beta_thresh], y_min, rho_thresh,
#                 alpha=0.07, color='firebrick',   zorder=1)   # ✗ resi  ✗ reli
# ax.fill_between([beta_thresh, x_max], y_min, rho_thresh,
#                 alpha=0.07, color='darkorange',  zorder=1)   # ✗ resi  ✓ reli

# ax.axvline(beta_thresh, color='gray', lw=0.8, ls=':', zorder=2)
# ax.axhline(rho_thresh,  color='gray', lw=0.8, ls=':', zorder=2)

# # Quadrant labels
# kw_q = dict(fontsize=8.5, alpha=0.8, va='center', ha='center', zorder=3)
# ax.text(0.5*(x_min+beta_thresh), 0.5*(rho_thresh+y_max),
#         '✓ resilient\n✗ reliable', **kw_q, color='forestgreen')
# ax.text(0.5*(beta_thresh+x_max), 0.5*(rho_thresh+y_max),
#         '✓ resilient\n✓ reliable', **kw_q, color='steelblue')
# ax.text(0.5*(x_min+beta_thresh), 0.5*(y_min+rho_thresh),
#         '✗ resilient\n✗ reliable', **kw_q, color='firebrick')
# ax.text(0.5*(beta_thresh+x_max), 0.5*(y_min+rho_thresh),
#         '✗ resilient\n✓ reliable', **kw_q, color='darkorange')

# ── Parametric curves ─────────────────────────────────────────────────────────
ax.plot(betas_bf, rhos_bf, color=col_bf, lw=2.0, alpha=0.9, zorder=3,
        label='Braced Frame (BF)')
ax.plot(betas_bi, rhos_bi, color=col_bi, lw=2.0, alpha=0.9, zorder=3,
        ls='--', label='Base Isolated (BI)')

# # Direction arrows at midpoint
# for bv, rv, col in [(betas_bf, rhos_bf, col_bf),
#                     (betas_bi, rhos_bi, col_bi)]:
#     mid = len(bv) // 2
#     if mid + 4 < len(bv):
#         ax.annotate('',
#             xy     =(bv[mid+4], rv[mid+4]),
#             xytext =(bv[mid],   rv[mid]),
#             arrowprops=dict(arrowstyle='->', color=col, lw=2.0),
#             zorder=8,
#         )

# Start (▲ low mu) / End (■ high mu) markers
for bv, rv, col in [(betas_bf, rhos_bf, col_bf),
                    (betas_bi, rhos_bi, col_bi)]:
    ax.scatter(bv[0],  rv[0],  s=180, marker='^', color=col,
               edgecolor='white', linewidth=0.8, zorder=7)
    ax.scatter(bv[-1], rv[-1], s=180, marker='s', color=col,
               edgecolor='white', linewidth=0.8, zorder=7)

# ── Reference points from results ─────────────────────────────────────────────
sc_markers = {'short': 's', 'long': '^'}
sc_labels  = {'short': 'short recovery', 'long': 'long recovery'}

for sc_label, mk in sc_markers.items():
    for st, col, Q_base, offset in [
            ('fixed',    col_bf, Q_bf_base, (+6, -14)),
            ('isolated', col_bi, Q_bi_base, (+6, +6 )),
    ]:
        b, r, mu = ref_pts[st][sc_label]
        ax.scatter(b, r, s=140, marker=mk, color=col,
                   edgecolor='white', linewidth=1.0, zorder=9)
        ax.annotate(
            f'{sc_label}\nμ={mu:.1f} yr⁻¹',
            (b, r), textcoords='offset points', xytext=offset,
            fontsize=7, color=col, ha='left',
        )

# # ── Colorbars ─────────────────────────────────────────────────────────────────
# for k, (col, cm, label) in enumerate([
#         (col_bf, cmap_bf, 'BF  $\\mu_{10}$ [yr$^{-1}$]'),
#         (col_bi, cmap_bi, 'BI  $\\mu_{10}$ [yr$^{-1}$]'),
# ]):
#     cax = fig.add_axes([0.76, 0.10 + k * 0.44, 0.018, 0.38])
#     cb  = fig.colorbar(ScalarMappable(norm=norm_mu, cmap=cm), cax=cax)
#     cb.set_label(label, color=col, fontsize=8)
#     ticks = np.arange(
#         int(np.floor(np.log10(mus_sweep.min()))),
#         int(np.ceil( np.log10(mus_sweep.max()))) + 1
#     )
#     cb.set_ticks(ticks)
#     cb.set_ticklabels([fr'$10^{{{int(t)}}}$' for t in ticks], fontsize=7.5)
#     plt.setp(cb.ax.yaxis.get_ticklabels(), color=col)

# ── Formatting ────────────────────────────────────────────────────────────────
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_xlabel(r'Reliability index $\beta_{50\mathrm{yr}}$')
ax.set_ylabel(r'Resilience index $\rho_{50\mathrm{yr}}$')
ax.set_title(
    r'$(\beta,\,\rho)$ plane — varying $\mu_{10}$' '\n'
    fr'$T_{{\mathrm{{hor}}}}={T_hor_plot:.0f}$ yr  |  '
    r'$\blacktriangle$ = low $\mu_{10}$,  $\blacksquare$ = high $\mu_{10}$',
    fontsize=10,
)
ax.grid(True, ls='--', alpha=0.3)

from matplotlib.lines import Line2D
leg = [
    Line2D([],[],color=col_bf, lw=2,    label='Braced Frame (BF)'),
    Line2D([],[],color=col_bi, lw=2, ls='--', label='Base Isolated (BI)'),
    Line2D([],[],marker='^', color='gray', ms=8, lw=0,
           markeredgecolor='white', label=r'low $\mu_{10}$'),
    Line2D([],[],marker='s', color='gray', ms=8, lw=0,
           markeredgecolor='white', label=r'high $\mu_{10}$'),
]
ax.legend(handles=leg, fontsize=8, loc='lower right', framealpha=0.85)

plt.tight_layout()

if general.get('savefig', False):
    fname = '06-beta-vs-rho-parametric.svg'
    plt.savefig(general['saveFolder'] / fname, transparent=True, format='svg', bbox_inches='tight')
    print(f'Saved: {fname}')

plt.show()

### Additional tables at T = 50 yr

In [ ]:
# ── Tabella beta e rho per T=50yr ────────────────────────────────────────────
T_hor_print = 50.0

print(f"\n{'='*65}")
print(f"  beta e rho  —  T = {T_hor_print:.0f} yr")
print(f"{'='*65}")
print(f"  {'System':<12} {'Recovery':<10} {'mu10 [yr⁻¹]':>12} {'beta':>8} {'rho':>8}")
print(f"  {'-'*12} {'-'*10} {'-'*12} {'-'*8} {'-'*8}")

for st, label in [('fixed', 'Braced Frame'), ('isolated', 'Base Isolated')]:
    for sc_label in ['short', 'long']:
        r = results[st][(sc_label, T_hor_print)]
        print(f"  {label:<12} {sc_label:<10} {r['mu10']:>12.2f} {r['beta']:>8.3f} {r['rho']:>8.3f}")
    print()

In [ ]:
# ── Tabella beta e rho per T=50yr — completa come Tabella 3 ──────────────────
from scipy.linalg import expm

T_hor_print = 50.0

# mu10 aggiuntivi dalla tabella
extra_mus = {
    'fixed'   : {'medium': 2.10},   # 120d
    'isolated': {'medium': 8.42},   # 30d
}

print(f"\n{'='*65}")
print(f"  beta e rho  —  T = {T_hor_print:.0f} yr")
print(f"{'='*65}")
print(f"  {'System':<16} {'Recovery':<10} {'tmed [days]':>11} {'mu10':>8} {'beta':>8} {'rho':>8}")
print(f"  {'-'*16} {'-'*10} {'-'*11} {'-'*8} {'-'*8} {'-'*8}")

tmed_map = {
    'fixed'   : {'short': 30,  'medium': 120, 'long': 365},
    'isolated': {'short':  1,  'medium':  30, 'long':  50},
}
mu_map = {
    'fixed'   : {'short': 8.43, 'medium': 2.10, 'long': 0.69},
    'isolated': {'short': 253., 'medium': 8.42, 'long': 5.06},
}
labels = {'fixed': 'Braced Frame', 'isolated': 'Base Isolated'}

for st in ['fixed', 'isolated']:
    Q_base = Qsim['ita'][hqtile][st][stqtile]['high'].copy()
    for sc in ['short', 'medium', 'long']:
        mu   = mu_map[st][sc]
        tmed = tmed_map[st][sc]
        Q    = core.set_recovery_rate(Q_base, mu)
        r    = core.reliability_resilience_summary(Q, T_hor_print)
        print(f"  {labels[st]:<16} {sc:<10} {tmed:>11d} {mu:>8.2f} {r['beta']:>8.3f} {r['rho']:>8.3f}")
    print()